# 03 — Coorte, painel, atributos e divisão temporal (seções 6–11 do v5)

**Lê:** `weekly_observed`, `weeks_info`, `corte.json` (01); `exogenas_semanais`, `exogenas_listas.json` (02).  
**Grava:** `coorte`, `panel_base`, `painel` (com a coluna `conjunto` = treino/teste/fora), `painel_spec.json`, `divisao.json`.

Este notebook **constrói** o teste mas não avalia nada nele. Daqui em diante, as linhas de teste só saem por `A.carregar_teste()`, que exige a seleção congelada.

In [1]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "03_painel"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

AVISO de reprodutibilidade: pandas=3.1.1; referência recomendada=2.2.3
AVISO de reprodutibilidade: numpy=2.5.4; referência recomendada=2.1.3
AVISO de reprodutibilidade: scikit-learn=1.9.1; referência recomendada=1.6.1
Notebook: 03_painel | execução: v6_modular | raiz: C:\Users\andre\OneDrive\Documentos\Projetos\Tccv2\tcc_fluxo_modular_v6


In [2]:
from tcc_fluxo import painel as P

corte = A.carregar_json("corte")
TEST_END = pd.Timestamp(corte["TEST_END"])
weekly_observed = A.carregar_df("weekly_observed")
wi = A.carregar_df("weeks_info").set_index("week_start")
exog = A.carregar_df("exogenas_semanais")
listas = A.carregar_json("exogenas_listas")

print("Semanas com aprovações antes de/até o início do painel:")
display(wi.loc[:C.PANEL_START + pd.Timedelta(weeks=3), ["pedidos", "valor"]])
pre = weekly_observed["week_start"] < C.PANEL_START
print(f"Registros vendedor-semana anteriores a {C.PANEL_START.date()} descartados: {pre.sum():,}")

Semanas com aprovações antes de/até o início do painel:


,pedidos,valor
week_start,,
2016-10-03,230,"45,052.0500"
2016-10-10,80,"12,044.1300"
2016-12-19,1,19.6200
2017-01-02,43,"3,649.8300"
2017-01-09,72,"13,164.7100"
2017-01-16,184,"27,912.4600"
2017-01-23,342,"62,395.5700"


Registros vendedor-semana anteriores a 2017-01-02 descartados: 181


## 6. Coorte (somente informação até `TRAIN_END`)

In [3]:
elig, cohort, COHORT_LABEL, observed_train = P.coorte(
    weekly_observed, C.PANEL_START, C.TRAIN_END, C.MIN_CALENDAR_WEEKS, C.MIN_SALES_WEEKS,
    C.USE_MEI_PROXY, C.MEI_MONTHLY_PROXY_LIMIT)
print("Critério:", COHORT_LABEL)
print(f"Vendedores com venda no treino: {len(elig):,} | elegíveis: {len(cohort):,} | excluídos: {len(elig) - len(cohort):,}")
fora = set(weekly_observed.loc[weekly_observed["week_start"] > C.TRAIN_END, "seller_id"]) - set(elig.index)
print(f"Vendedores com venda no teste e sem venda no treino (fora da coorte por construção): {len(fora):,}")
display(cohort[["avg_weekly_revenue_train", "avg_monthly_revenue_train", "revenue_train", "sales_weeks",
                "calendar_weeks"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).T)

Critério: vendedores elegíveis segundo o critério semanal definido
Vendedores com venda no treino: 1,784 | elegíveis: 870 | excluídos: 914
Vendedores com venda no teste e sem venda no treino (fora da coorte por construção): 1,281


,count,mean,std,min,25%,50%,75%,90%,max
avg_weekly_revenue_train,870.0000,198.0851,482.0601,2.1464,35.2502,84.8068,189.6502,408.2812,"10,530.1306"
avg_monthly_revenue_train,870.0000,861.3163,"2,096.1006",9.3332,153.2756,368.7581,824.6398,"1,775.2943","45,787.2641"
revenue_train,870.0000,"7,392.1528","15,912.1995",96.5900,"1,194.5125","2,868.9750","6,926.6950","15,810.3423","189,542.3500"
sales_weeks,870.0000,16.4264,11.7550,4.0000,7.0000,13.0000,23.0000,35.0000,52.0000
calendar_weeks,870.0000,31.9805,12.2653,12.0000,21.0000,33.0000,43.0000,48.0000,52.0000


## 7. Painel vendedor-semana (zeros preenchidos, sem interpolação)

In [4]:
panel_base = P.montar_panel_base(cohort, weekly_observed, TEST_END, C.PANEL_START)
print(f"Linhas: {len(panel_base):,} | vendedores: {panel_base['seller_id'].nunique():,} | "
      f"semanas: {panel_base['week_start'].nunique()}")
print(f"Semanas com receita zero: {panel_base['revenue'].eq(0).mean():.1%}")

Linhas: 61,857 | vendedores: 870 | semanas: 86
Semanas com receita zero: 61.4%


## 8–10. Atributos sem vazamento, calendário da semana-alvo e alvo (h = 1 semana)

In [5]:
panel = P.attach_exogenous_features(panel_base, exog)
EXOG = listas["EXOGENOUS_FEATURES"]
ALL_NUMERIC = P.BASE_NUMERIC_FEATURES + EXOG
assert ((panel["target_week"] - panel["week_start"]).dropna() == pd.Timedelta(days=7)).all()
assert panel.loc[panel["target_week"].notna(), EXOG].notna().all().all()

feature_cols = ALL_NUMERIC + ["target_week", "target_week_of_year"]
_, changed = P.teste_vazamento(panel_base, panel, exog, feature_cols, C.LEAK_TEST_CUT, C.RANDOM_SEED)
print(f"Atributos até {C.LEAK_TEST_CUT.date()} idênticos após perturbar o futuro: OK "
      f"(controle: {changed:.0%} dos alvos após o corte mudaram)")

Atributos até 2017-09-04 idênticos após perturbar o futuro: OK (controle: 100% dos alvos após o corte mudaram)


## 11. Divisão temporal pela data do alvo

In [6]:
painel, resumo = P.dividir(panel, ALL_NUMERIC, C.TRAIN_TARGET_END, TEST_END)
print(pd.Series(resumo))
profile = P.perfis_treino(cohort)
print(profile.groupby("perfil", observed=True)["freq_venda_treino"].agg(["min", "max", "size"]))

linhas_com_alvo                          60987
linhas_utilizaveis                       54027
treino_linhas                            24447
treino_vendedores                          870
treino_alvos          [2017-03-06, 2017-12-25]
treino_semanas                              43
teste_linhas                             29580
teste_vendedores                           870
teste_alvos           [2018-01-01, 2018-08-20]
teste_semanas                               34
alvos_zero_treino                       0.5735
alvos_zero_teste                        0.6762
dtype: object
                min    max  size
perfil                          
intermitente 0.0909 0.3421   290
medio        0.3438 0.6000   292
regular      0.6061 1.0000   288


In [7]:
A.salvar_df(cohort.reset_index(), "coorte")
A.salvar_df(profile.reset_index(), "perfis_treino")
A.salvar_df(panel_base, "panel_base")
A.salvar_df(painel, "painel")
A.salvar_json({"BASE_NUMERIC_FEATURES": P.BASE_NUMERIC_FEATURES, "EXOGENOUS_FEATURES": EXOG,
               "CALENDAR_FEATURES": listas["CALENDAR_FEATURES"], "HOLIDAY_FEATURES": listas["HOLIDAY_FEATURES"],
               "COMMERCIAL_FEATURES": listas["COMMERCIAL_FEATURES"], "MACRO_FEATURES": listas["MACRO_FEATURES"],
               "CAT_FEATURES": P.CAT_FEATURES, "TARGET": P.TARGET, "COHORT_LABEL": COHORT_LABEL}, "painel_spec")
A.salvar_json({**resumo, "TEST_END": TEST_END, "teste_vazamento_corte": C.LEAK_TEST_CUT,
               "teste_vazamento_controle": changed}, "divisao")

{'arquivo': 'divisao.json',
 'sha256': '25c15f889ad5470cd4d09b00b516e8e45f70386815233d24a68769452d64fcb9',
 'notebook': '03_painel',
 'criado_em': '2026-11-06T20:21:33',
 'entradas': {'corte': 'c778cdb2b687163abee02bfdc83352d505599df8cd531f925b3254f7fc2d2271',
  'weekly_observed': 'edce3925d398cba7a90afd80cef9d9942c18ac355511e9ef2d2d872742e96294',
  'weeks_info': 'e5ad16d16cf8d112dc4fd949cfcf9f2c170a9347918f8b8561c4e8b960205e95',
  'exogenas_semanais': '639fda267b2a1cd7af3d3e224f9b3070add129c69ee7e371bb896e509b1aee03',
  'exogenas_listas': '6799286410608f4f80d74c0873c08bc2d667302522f1fb3bd033f3bd374a6aa1'}}

In [8]:
A.finalizar()

03_painel: leu 5 artefato(s), gravou 6.
